# Prov-GigaPath features for colon_n (Colab, T4)
Runs the **unmodified** upstream `extract_features.py` (GeorgeBatch/LC25000-clean @ 912f342) for
`prov-gigapath`, `imagenet` normalisation, fp32, on colon_n. Press *Runtime > Run all* on a T4 GPU runtime.
Needs: `MyDrive/btp_repro/colon_n_original.zip`, Colab secret `HF_TOKEN` (with access to prov-gigapath/prov-gigapath).
Output: `MyDrive/btp_repro/gigapath_out/{features.npy,ids.npy,ids_2_img_paths.json}`.

In [ ]:
# 1. GPU check + Drive
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
import os
ZIP = '/content/drive/MyDrive/btp_repro/colon_n_original.zip'
OUT = '/content/drive/MyDrive/btp_repro/gigapath_out'
assert os.path.isfile(ZIP), f"missing {ZIP}"
print("zip bytes:", os.path.getsize(ZIP))

In [ ]:
# 2. HF token from Colab secrets (never printed)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], "HF_TOKEN secret is empty (enable 'Notebook access' for it)"
from huggingface_hub import HfApi
HfApi(token=os.environ['HF_TOKEN']).model_info('prov-gigapath/prov-gigapath')   # raises if no access
print("HF access to prov-gigapath/prov-gigapath: OK")

In [ ]:
# 3. Clone upstream at the pinned commit
COMMIT = '912f342fe2a781e847f395d79982dad4d3d425ae'
!rm -rf /content/LC25000-clean
!git clone -q https://github.com/GeorgeBatch/LC25000-clean /content/LC25000-clean
!git -C /content/LC25000-clean checkout -q {COMMIT}
import subprocess
head = subprocess.check_output(['git','-C','/content/LC25000-clean','rev-parse','HEAD']).decode().strip()
assert head == COMMIT, head
print("upstream HEAD:", head)
%cd /content/LC25000-clean

In [ ]:
# 4. Dependencies (keep Colab's torch/torchvision; timm>=1.0.3 is needed for Prov-GigaPath)
!pip install -q "timm>=1.0.3" huggingface_hub safetensors
import torch, torchvision, timm
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__)
assert tuple(int(x) for x in timm.__version__.split('.')[:3]) >= (1, 0, 3)

In [ ]:
# 5. Unzip images to LC25000/colon_n and check
import glob
!mkdir -p /content/LC25000-clean/LC25000/colon_n
!unzip -q -o "{ZIP}" -d /content/LC25000-clean/LC25000/colon_n
n = len(glob.glob('/content/LC25000-clean/LC25000/colon_n/*.jpeg'))
assert n == 5000, n
print("jpeg files:", n)

In [ ]:
# 6. Extract features: unmodified upstream script, fp32 (no --precision half)
!python extract_features.py --cancer_type colon_n --img_norm imagenet --extractor_name prov-gigapath --device cuda --batch_size 32

In [ ]:
# 7. Check, copy to Drive, hash
import numpy as np, hashlib, shutil, json
FD = '/content/LC25000-clean/feature_vectors/colon_n/prov-gigapath/imagenet'
feats = np.load(f'{FD}/features.npy')
print("features:", feats.shape, feats.dtype, "| finite:", bool(np.isfinite(feats).all()))
assert feats.shape == (5000, 1536), feats.shape
ids2p = json.load(open(f'{FD}/ids_2_img_paths.json'))
assert len(ids2p) == 5000 and len(set(ids2p.values())) == 5000
os.makedirs(OUT, exist_ok=True)
for f in ['features.npy', 'ids.npy', 'ids_2_img_paths.json']:
    shutil.copy(f'{FD}/{f}', f'{OUT}/{f}')
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| timm", timm.__version__)
for f in ['features.npy', 'ids.npy', 'ids_2_img_paths.json']:
    print(hashlib.sha256(open(f'{OUT}/{f}', 'rb').read()).hexdigest(), f)
print("copied to", OUT)